In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import torch

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from src.trainer import InterContiNetTrainer, SimpleTrainer
from src.data_utils import get_mnist_tasks, _extract_targets, get_context_sets
from src.utils import InContextHead
from src import models
from src.regulariser import UnbiasRegulariser, L2Regulariser, MultiRegulariser

### Task Incremental Learning

In [4]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cuda")
head.set_context(0)
model = models.get_mnist_model(head=head, device="cuda")

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[3, 4], [8, 9], [6, 7], [0, 5], [1, 2]]


In [5]:
trainer = SimpleTrainer(model)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

Training Epochs:   0%|                                                                   | 0/3 [00:00<?, ?it/s, val_loss=N\A]

Training Epochs: 100%|█████████████████████████████████████████| 3/3 [00:18<00:00,  6.06s/it, val_loss=0.0369, val_acc=0.991]


Test Results: [(0.0209, 0.993)]


[(0.0209, 0.993)]

In [19]:
interval_trainer = InterContiNetTrainer(
    trainer.model,
    min_acc_limit=0.9,
    paradigm="TIL",
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0], context_id=0)

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    interval_trainer.train(train, val, batch_size=256, context_id=i)
    test_vals = interval_trainer.test(test_tasks[0 : i + 1], context_list=list(range(i + 1)))
    if i < len(train_tasks) - 2:
        target_acc = min(max(test_vals[-1][1] - interval_trainer.min_acc_increment, 0), interval_trainer.min_acc_limit)
        interval_trainer.min_acc_limit = target_acc
        interval_trainer.compute_rashomon_set(test, context_id=i)

  0%|                                                                                                | 0/100 [00:00<?, ?it/s]

 93%|█████████████████████████████████████████████████████    | 93/100 [00:39<00:02,  2.38it/s, max loss=0.1139, min acc=1.0]


LID size: 234.0977


Training Epochs: 100%|████████████████████████████████████████| 5/5 [00:28<00:00,  5.78s/it, val_loss=0.1054, val_acc=0.9702]


Test Results: [(0.0204, 0.993), (0.0823, 0.9763)]


...

LID size: 34.9722


Training Epochs: 100%|████████████████████████████████████████| 5/5 [00:26<00:00,  5.33s/it, val_loss=0.3278, val_acc=0.9072]


Test Results: [(0.0204, 0.993), (0.0827, 0.9768), (0.0281, 0.9909), (0.2894, 0.9274)]


Training Epochs: 100%|████████████████████████████████████████| 5/5 [00:33<00:00,  6.74s/it, val_loss=0.2036, val_acc=0.9581]


Test Results: [(0.0204, 0.993), (0.0826, 0.9758), (0.0298, 0.9894), (0.3009, 0.9193), (0.2116, 0.9677)]


### Domain Incremental Learning

In [20]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = models.get_mnist_model(device="cuda")

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[3, 4], [8, 9], [6, 7], [0, 5], [1, 2]]


In [21]:
def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
    """Map the global label to the in context label."""
    return labels % 2

In [22]:
trainer = SimpleTrainer(model, paradigm="DIL", domain_map_fn=domain_map_fn)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

Training Epochs:   0%|                                                                   | 0/3 [00:00<?, ?it/s, val_loss=N\A]

Training Epochs: 100%|█████████████████████████████████████████| 3/3 [00:17<00:00,  5.98s/it, val_loss=0.0352, val_acc=0.991]


Test Results: [(0.0208, 0.994)]


[(0.0208, 0.994)]

In [26]:
interval_trainer = InterContiNetTrainer(
    trainer.model,
    min_acc_limit=0.7,
    paradigm="DIL",
    domain_map_fn=domain_map_fn
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0])

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    interval_trainer.train(train, val, batch_size=256)
    test_vals = interval_trainer.test(test_tasks[0 : i + 1])
    if i < len(train_tasks) - 2:
        target_acc = min(max(test_vals[-1][1] - interval_trainer.min_acc_increment, 0), interval_trainer.min_acc_limit)
        interval_trainer.min_acc_limit = target_acc
        interval_trainer.compute_rashomon_set(test)

 42%|███████████████████████▌                                | 42/100 [00:16<00:22,  2.57it/s, max loss=0.9067, min acc=0.75]


LID size: 66.4201


Training Epochs: 100%|████████████████████████████████████████| 5/5 [00:28<00:00,  5.70s/it, val_loss=2.7076, val_acc=0.1851]


Test Results: [(0.0241, 0.994), (2.8118, 0.1765)]


  5%|██▊                                                    | 5/100 [00:01<00:37,  2.54it/s, max loss=3.8866, min acc=0.1406]


...

LID size: 15.5754


Training Epochs: 100%|████████████████████████████████████████| 5/5 [00:26<00:00,  5.29s/it, val_loss=3.7945, val_acc=0.3882]


Test Results: [(0.0244, 0.9935), (2.7179, 0.1841), (0.3586, 0.8444), (3.5666, 0.4017)]


Training Epochs: 100%|████████████████████████████████████████| 5/5 [00:33<00:00,  6.66s/it, val_loss=2.3210, val_acc=0.5812]


Test Results: [(0.0256, 0.993), (2.6905, 0.1962), (0.3731, 0.8429), (3.3699, 0.3996), (2.7063, 0.5611)]


### Class Incremental Training

In [27]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[3, 4], [8, 9], [6, 7], [0, 5], [1, 2]]


In [28]:
trainer = SimpleTrainer(model)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

Training Epochs:   0%|                                                                   | 0/3 [00:00<?, ?it/s, val_loss=N\A]

Training Epochs: 100%|█████████████████████████████████████████| 3/3 [00:17<00:00,  5.89s/it, val_loss=0.0363, val_acc=0.991]


Test Results: [(0.0205, 0.9935)]


[(0.0205, 0.9935)]

In [30]:
interval_trainer = InterContiNetTrainer(
    trainer.model,
    min_acc_limit=0.7,
    paradigm="CIL",
)

# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0], target_acc=0.7)

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    interval_trainer.train(train, val, batch_size=256)
    test_vals = interval_trainer.test(test_tasks[0 : i + 1])
    if i < len(train_tasks) - 2:
        target_acc = min(max(test_vals[-1][1] - interval_trainer.min_acc_increment, 0), interval_trainer.min_acc_limit)
        interval_trainer.min_acc_limit = target_acc
        interval_trainer.compute_rashomon_set(test, target_acc=target_acc)

 42%|███████████████████████▌                                | 42/100 [00:17<00:24,  2.39it/s, max loss=0.9250, min acc=0.75]


LID size: 71.4827


Training Epochs: 100%|███████████████████████████████████████| 5/5 [00:28<00:00,  5.75s/it, val_loss=23.7345, val_acc=0.0000]


Test Results: [(0.0219, 0.9935), (24.3746, 0.0)]


  0%|                                                                 | 0/100 [00:00<?, ?it/s, max loss=31.2675, min acc=0.0]


...

LID size: 17.7452


Training Epochs: 100%|███████████████████████████████████████| 5/5 [00:26<00:00,  5.31s/it, val_loss=29.2639, val_acc=0.0000]


Test Results: [(0.0234, 0.9935), (24.0285, 0.0), (22.9235, 0.0), (29.5183, 0.0)]


Training Epochs: 100%|███████████████████████████████████████| 5/5 [00:33<00:00,  6.67s/it, val_loss=21.3542, val_acc=0.0000]


Test Results: [(0.024, 0.9935), (24.3037, 0.0), (23.296, 0.0), (30.0059, 0.0), (21.8735, 0.0)]
